Notebook to produce the final table and visualisation that were sent to the team in March 2025.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import umap

from collections import defaultdict
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from typing import List, Iterator, Dict, Callable, Type, Tuple

nltk.download("wordnet")

LEMMATIZER = WordNetLemmatizer()

from dsp_ai_eval import PROJECT_DIR

In [ ]:
# Functions and constants
NESTA_COLOURS = [
    "#0000FF",
    "#FDB633",
    "#18A48C",
    "#9A1BBE",
    "#EB003B",
    "#FF6E47",
    "#646363",
    "#0F294A",
    "#97D9E3",
    "#A59BEE",
    "#F6A4B7",
    "#D2C9C0",
    # "#FFFFFF",
    "#000000",
]


def stacked_bar(counts_df, x, y, colour):
    # Create a stacked bar chart
    fig = px.bar(
        counts_df,
        x=x, 
        y=y,
        color=colour,
        orientation='h',  # Horizontal bars
        barmode='stack'  # Stacked bars
    )
    
    fig.update_layout(
        width=1000,
        height=800,
    )

    return fig

def scatterplot(data_viz, outpath='topic_scatterplot_rq2.html'):
    """Default scatterplot that was sent initially"""
    fig = px.scatter(
        data_viz,
        x="x",
        y="y",
        # size="total_cites",
        color="topic_name",
        hover_data={"topic_name": True, "title": True, "publication_year": True, "total_cites": True, 
                    "doi": True, 
                    "x": False, "y": False},
        color_discrete_sequence=NESTA_COLOURS,
        opacity=0.5,
    )

    fig.update_layout(
            width=1200,  # Increase width
            height=800,  # Adjust height
            xaxis=dict(showticklabels=False, title_text=""),  # Hide x-axis ticks and title
            yaxis=dict(showticklabels=False, title_text=""),  # Hide y-axis ticks and title
            legend_title_text="",  # Hide legend title
            plot_bgcolor="white",  # Set background of the plot area to white
            paper_bgcolor="white",  # Set background of the entire figure to white
        )

    fig.write_html(outpath)

    return fig


### Functions taken and modified from discovery_utils ###

def simple_tokenizer(text: str) -> List[str]:
    """Split the text into words"""
    return text.split()


def preproc(text: str, list_of_stopwords: List[str] = stopwords.words("english")) -> str:
    """Preprocess text by removing non-alphabetic characters, lowercasing, lemmatising, and removing stopwords"""
    text = re.sub(r"[^a-zA-Z ]+", "", text).lower()
    text = text.split()
    text = [LEMMATIZER.lemmatize(t) for t in text]
    text = [t for t in text if t not in list_of_stopwords]
    return " ".join(text)

def concat_texts_in_cluster(documents: Iterator[str], cluster_labels: Iterator) -> Dict:
    """
    Create a large text string for each cluster, by joining up the text strings (documents) belonging to the same cluster

    Args:
        documents: A list of text strings
        cluster_labels: A list of cluster labels, indicating the membership of the text strings
    Returns:
        A dictionary where keys are cluster labels, and values are cluster text documents
    """

    assert len(documents) == len(cluster_labels)
    doc_type = type(documents[0])

    cluster_text_dict = defaultdict(doc_type)
    for i, doc in enumerate(documents):
        if doc_type is str:
            cluster_text_dict[cluster_labels[i]] += doc + " "
        elif doc_type is list:
            cluster_text_dict[cluster_labels[i]] += doc
    return cluster_text_dict

def generate_cluster_keywords(
    documents: Iterator[str],
    cluster_labels: Iterator[int],
    n: int = 10,
    tokenizer: Callable = simple_tokenizer,
    max_df: float = 0.90,
    min_df: float = 0.01,
    Vectorizer: Type[TfidfVectorizer] = TfidfVectorizer,
) -> Dict:
    """
    Generate keywords that characterise the cluster, using the specified Vectorizer

    Args:
        documents: List of (preprocessed) text documents
        cluster_labels: List of integer cluster labels
        n: Number of top keywords to return
        Vectorizer: Vectorizer object to use (eg, TfidfVectorizer, CountVectorizer)
        tokenizer: Function to use to tokenise the input documents; by default splits the document into words
    Returns:
        Dictionary that maps cluster integer labels to a list of keywords
    """

    # Define vectorizer
    vectorizer = Vectorizer(
        analyzer="word",
        tokenizer=tokenizer,
        preprocessor=lambda x: x,
        token_pattern=None,
        max_df=max_df,
        min_df=min_df,
        max_features=10000,
    )

    # Create cluster text documents
    cluster_documents = concat_texts_in_cluster(documents, cluster_labels)
    unique_cluster_labels = list(cluster_documents.keys())

    # Apply the vectorizer
    token_score_matrix = vectorizer.fit_transform(list(cluster_documents.values()))

    # Create a token lookup dictionary
    id_to_token = dict(zip(list(vectorizer.vocabulary_.values()), list(vectorizer.vocabulary_.keys())))

    # For each cluster, check the top n tokens
    top_cluster_tokens = {}
    for i in range(token_score_matrix.shape[0]):
        # Get the cluster feature vector
        x = token_score_matrix[i, :].todense()
        # Find the indices of the top n tokens
        x = list(np.flip(np.argsort(np.array(x)))[0])[0:n]
        # Find the tokens corresponding to the top n indices
        top_cluster_tokens[unique_cluster_labels[i]] = [id_to_token[j] for j in x]

    return top_cluster_tokens

def generate_landscape_keywords(
    viz_df: pd.DataFrame,
    n_keyword_clusters: int = 10,
    random_state: int = 42,
    x_col: str = "umap_x",
    y_col: str = "umap_y",
    text_col: str = "text",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate keywords for the landscape"""
    clusterer = KMeans(n_clusters=n_keyword_clusters, random_state=random_state)
    clusterer.fit(viz_df[[x_col, y_col]])
    soft_clusters = list(clusterer.labels_)

    title_texts = viz_df[text_col].apply(preproc)
    _cluster_texts = concat_texts_in_cluster(title_texts, soft_clusters)
    _cluster_keywords = generate_cluster_keywords(
        documents=list(_cluster_texts.values()),
        cluster_labels=list(_cluster_texts.keys()),
        n=2,
        max_df=0.90,
        min_df=0.01,
        Vectorizer=TfidfVectorizer,
    )
    viz_df["soft_cluster"] = soft_clusters
    viz_df["soft_cluster_"] = [str(x) for x in soft_clusters]
    viz_df["keyword_cluster"] = viz_df["soft_cluster"].apply(lambda x: ", ".join(_cluster_keywords[x]))

    centroids = (
        viz_df.groupby("soft_cluster")
        .agg(x_c=(x_col, "mean"), y_c=(y_col, "mean"))
        .reset_index()
        .assign(keywords=lambda x: x.soft_cluster.apply(lambda y: ", ".join(_cluster_keywords[y])))
    )

    return viz_df, centroids

In [ ]:
data_viz = pd.read_parquet('s3://dsp-ai-eval/racial_bias_org_change/rq2/outputs/openalex/data/visualization_data.parquet')

In [ ]:
# Table output: https://docs.google.com/spreadsheets/d/12ABDqjWAGunKZA06jHoVUO_SvNCa1osAjB_4CHLZqIc/edit?gid=157534602#gid=157534602
output_data = data_viz[['id', 'doi', 'title', 'abstract', 'publication_year','total_cites', 'referenced_works', 'related_works', 'journal',
          'topic_name', 'topic_description', 'topic_keywords', 'evidence_type', 'dei', 'justice']]
output_data.to_csv('rq2_llm_aided_output.csv', index=False)

In [ ]:
output_data.groupby(['topic_name', 'dei']).size()

topic_dei_mentions = output_data.groupby(['topic_name', 'dei']).size().reset_index(name='count')

stacked_bar(topic_dei_mentions, x='count', y='topic_name', colour='dei')

In [ ]:
relevant_topics = ['Evolving Police Culture and Reform',
                   'Middle Management and Organizational Change',
                   'Psychological Dynamics of Organizational Change',
                   'Organizational Change and Cultural Dynamics',
                   'Navigating Workplace Diversity',
                   ]


In [ ]:
embeddings_array = np.vstack(data_viz['embeddings'].values)

# Apply UMAP
reducer = umap.UMAP(n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(embeddings_array)

# Add results to dataframe
data_viz[['x', 'y']] = embedding_2d

In [ ]:
data_viz_copy = data_viz[data_viz['topic_name'].isin(relevant_topics)].copy()

data_viz_copy, centroids = generate_landscape_keywords(
    data_viz_copy,
    n_keyword_clusters = 12,
    random_state = 42,
    x_col = "x",
    y_col = "y",
    text_col= "title_abstract",
)

In [ ]:
duplicates = data_viz_copy.duplicated(subset=["soft_cluster", "soft_cluster_", "keyword_cluster"], keep="first")

data_viz_copy.loc[duplicates, "keyword_cluster"] = ""
data_viz_copy["keyword_cluster"] = data_viz_copy["keyword_cluster"].fillna("")

In [ ]:
fig = px.scatter(
        data_viz_copy,
        x="x",
        y="y",
        text="keyword_cluster",
        color="topic_name",
        # hover_data=["topic_name", "title", "doi"],
        hover_data={"topic_name": True, "title": True, "doi": True,
                    "total_cites": True, "publication_year": True,
                    "keyword_cluster": False,
                    "x": False, "y": False},
        custom_data=["topic_name", "title_abstract"],
        color_discrete_sequence=NESTA_COLOURS,
        opacity=0.3,
        #category_orders={"Name": sorted_list},
    )

fig.update_traces(
        textposition="top center",
        textfont=dict(size=14, color="black"),  # Increase font size  # Text color
    )

fig.update_layout(
        width=1200,  # Increase width
        height=800,  # Adjust height
        xaxis=dict(showticklabels=False, title_text=""),  # Hide x-axis ticks and title
        yaxis=dict(showticklabels=False, title_text=""),  # Hide y-axis ticks and title
        legend_title_text="",  # Hide legend title
        plot_bgcolor="white",  # Set background of the plot area to white
        paper_bgcolor="white",  # Set background of the entire figure to white
    )

fig.write_html(PROJECT_DIR / 'outputs/racial_bias_org_change/topic_scatterplot_rq2_v2.html')

fig.show()

In [ ]:
# plot v1
from dsp_ai_eval import PROJECT_DIR

fig = scatterplot(data_viz, PROJECT_DIR / 'outputs/racial_bias_org_change/topic_scatterplot_rq2.html')
fig.show()

# Identify abstracts that mention specific types of evidence

In [ ]:
cluster_summaries = pd.read_parquet('s3://dsp-ai-eval/racial_bias_org_change/rq2/outputs/openalex/data/abstracts_cluster_summaries_cleaned.parquet')

In [ ]:
cluster_summaries.head()

In [ ]:
cluster_summaries.to_csv('rq2_cluster_summaries.csv')

In [ ]:
fig = px.scatter(
        data_viz[data_viz['mentions_evidence_type']==True],
        x="x",
        y="y",
        # text="keywords",
        color="topic_name",
        hover_data={"topic_name": True, "title": True, "publication_year": True, "total_cites": True, 
                    # "journal": True, 
                    "x": False, "y": False},
        color_discrete_sequence=NESTA_COLOURS,
        opacity=0.5,
    )

fig.update_layout(
        width=1200,  # Increase width
        height=800,  # Adjust height
        xaxis=dict(showticklabels=False, title_text=""),  # Hide x-axis ticks and title
        yaxis=dict(showticklabels=False, title_text=""),  # Hide y-axis ticks and title
        legend_title_text="",  # Hide legend title
        plot_bgcolor="white",  # Set background of the plot area to white
        paper_bgcolor="white",  # Set background of the entire figure to white
    )

fig.write_html('topic_scatterplot_rq1_evidence.html')

fig

In [ ]:
symbol_map = {True: 'circle', False: 'square'}

data_viz['symbol'] = data_viz['mentions_evidence_type'].map(symbol_map)

In [ ]:

fig = px.scatter(
    data_viz,
    x="x",
    y="y",
    color="topic_name",
    symbol="mentions_evidence_type",  # Different shapes for mentions_evidence
    hover_data={
        "topic_name": True,
        "title": True,
        "publication_year": True,
        "total_cites": True,
        # "journal": True,
        "x": False,
        "y": False
    },
    color_discrete_sequence=NESTA_COLOURS,
    opacity=0.3,
)

fig.update_layout(
    width=1200,
    height=800,
    xaxis=dict(showticklabels=False, title_text=""),
    yaxis=dict(showticklabels=False, title_text=""),
    legend_title_text="",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()


# Identifying which search terms contribute to which topics

In [ ]:
crosstab_result = pd.crosstab(data_viz['search_term'], data_viz['topic_name'])
crosstab_result

In [ ]:
crosstab_result.to_csv('search_term_topic_crosstab.csv')